In [37]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.sql import Window

In [38]:
# Create SparkSession
spark = (SparkSession.builder
                    .appName('PySparkSyntax')
                    .getOrCreate()
        )

spark.sparkContext.setLogLevel("WARN") # INFO WARN ERROR DEBUG A

In [42]:
# Define the schema for a DataFrame
schema = StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("city", StringType(), True)
])

In [43]:
# Create a DataFrame using the schema
data = [("Alice", 25, "New York")
        , ("Bob", 30, "San Francisco")
        , ("Bob", 12, "Las Vegas")
        , ("Charlie", 35, "Chicago")
        , ("Charlie", 35, "Chicago")]
df = spark.createDataFrame(data, schema)

In [6]:
# Show the DataFrame
df.show()

+-------+---+-------------+
|   name|age|         city|
+-------+---+-------------+
|  Alice| 25|     New York|
|    Bob| 30|San Francisco|
|    Bob| 12|    Las Vegas|
|Charlie| 35|      Chicago|
|Charlie| 35|      Chicago|
+-------+---+-------------+



# dropDuplicates

In [7]:
df.show()

+-------+---+-------------+
|   name|age|         city|
+-------+---+-------------+
|  Alice| 25|     New York|
|    Bob| 30|San Francisco|
|    Bob| 12|    Las Vegas|
|Charlie| 35|      Chicago|
|Charlie| 35|      Chicago|
+-------+---+-------------+



In [8]:
df_dp = df.dropDuplicates()

In [9]:
df_dp.show()

+-------+---+-------------+
|   name|age|         city|
+-------+---+-------------+
|  Alice| 25|     New York|
|    Bob| 30|San Francisco|
|Charlie| 35|      Chicago|
|    Bob| 12|    Las Vegas|
+-------+---+-------------+



In [10]:
df_dp = df.dropDuplicates(['name'])

In [11]:
df_dp.show()

+-------+---+---------+
|   name|age|     city|
+-------+---+---------+
|  Alice| 25| New York|
|    Bob| 12|Las Vegas|
|Charlie| 35|  Chicago|
+-------+---+---------+



In [22]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

In [23]:
window = Window.partitionBy("name").orderBy("age")

In [31]:
df.withColumn("rn", row_number().over(window)).show()

+-------+---+-------------+---+
|   name|age|         city| rn|
+-------+---+-------------+---+
|  Alice| 25|     New York|  1|
|    Bob| 12|    Las Vegas|  1|
|    Bob| 30|San Francisco|  2|
|Charlie| 35|      Chicago|  1|
|Charlie| 35|      Chicago|  2|
+-------+---+-------------+---+



In [30]:
df.withColumn("rn", row_number().over(window)).filter("rn = 1").show()

+-------+---+---------+---+
|   name|age|     city| rn|
+-------+---+---------+---+
|  Alice| 25| New York|  1|
|    Bob| 12|Las Vegas|  1|
|Charlie| 35|  Chicago|  1|
+-------+---+---------+---+



In [24]:
df_final = (df
            .withColumn("rn", row_number().over(window))
            .filter("rn = 1")
            .drop("rn"))

In [25]:
df_final.show()   # <- aqui sim

+-------+---+---------+
|   name|age|     city|
+-------+---+---------+
|  Alice| 25| New York|
|    Bob| 12|Las Vegas|
|Charlie| 35|  Chicago|
+-------+---+---------+



In [45]:
df.show()

+-------+---+-------------+
|   name|age|         city|
+-------+---+-------------+
|  Alice| 25|     New York|
|    Bob| 30|San Francisco|
|    Bob| 12|    Las Vegas|
|Charlie| 35|      Chicago|
|Charlie| 35|      Chicago|
+-------+---+-------------+



In [46]:
df_d = df.distinct()

In [47]:
df_d.show()

+-------+---+-------------+
|   name|age|         city|
+-------+---+-------------+
|Charlie| 35|      Chicago|
|    Bob| 12|    Las Vegas|
|  Alice| 25|     New York|
|    Bob| 30|San Francisco|
+-------+---+-------------+



In [48]:
df_d.toPandas()

,name,age,city
0,Alice,25,New York
1,Bob,30,San Francisco
2,Charlie,35,Chicago
3,Bob,12,Las Vegas


26/02/26 21:38:47 ERROR StandaloneSchedulerBackend: Application has been killed. Reason: Master removed our application: KILLED
26/02/26 21:38:47 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exiting due to error from cluster scheduler: Master removed our application: KILLED
	at org.apache.spark.errors.SparkCoreErrors$.clusterSchedulerError(SparkCoreErrors.scala:291)
	at org.apache.spark.scheduler.TaskSchedulerImpl.error(TaskSchedulerImpl.scala:981)
	at org.apache.spark.scheduler.cluster.StandaloneSchedulerBackend.dead(StandaloneSchedulerBackend.scala:165)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint.markDead(StandaloneAppClient.scala:263)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint$$anonfun$receive$1.applyOrElse(StandaloneAppClient.scala:170)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:115)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:213)
	at org.apache.spark.rpc.netty.Inbox.proce